In [2]:
# =========================================================
# 1. MOUNT DRIVE + INSTALL
# =========================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install catboost xgboost lightgbm -q

# =========================================================
# 2. IMPORTS + CONFIG
# =========================================================
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
from scipy.optimize import minimize

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# ----- CONFIG -----
TARGET   = 'demand'
N_SPLITS = 5
SEED     = 42
USE_LOG  = True   # try True AND False; keep whichever prints the higher OOF R2

print('Setup complete')

# =========================================================
# 3. LOAD DATA + DROP LEAKY / ID COLUMNS
# =========================================================
train_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/train_feature_engineered.csv'
test_path  = '/content/drive/MyDrive/Traffic_Prediction/dataset/test_feature_engineered.csv'

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)
print('Train:', train_df.shape, ' Test:', test_df.shape)

# Drop the leaky demand-mean features (replaced by OOF target encoding below)
leaky_cols = ['geohash_demand_mean', 'hour_demand_mean',
              'roadtype_demand_mean', 'day_demand_mean']
train_df.drop(columns=leaky_cols, inplace=True, errors='ignore')
test_df.drop(columns=leaky_cols,  inplace=True, errors='ignore')

# Drop Index from features; drop demand from test if present
for c in ['Index']:
    train_df.drop(columns=c, inplace=True, errors='ignore')
    test_df.drop(columns=c,  inplace=True, errors='ignore')
test_df.drop(columns='demand', inplace=True, errors='ignore')

print('Leaky / ID columns removed')

# =========================================================
# 4. LABEL-ENCODE CATEGORICALS (consistent across train+test)
# =========================================================
categorical_features = [
    'geohash', 'RoadType', 'LargeVehicles', 'Landmarks',
    'Weather', 'geohash_4', 'geohash_5', 'weather_temp_interaction'
]

# Columns to OOF target-encode (leak-free, computed inside each fold)
te_cols  = ['geohash', 'hour', 'RoadType', 'day']
te_names = [c + '_te' for c in te_cols]

for col in categorical_features:
    train_df[col] = train_df[col].astype(str)
    test_df[col]  = test_df[col].astype(str)
    le = LabelEncoder()
    le.fit(list(train_df[col]) + list(test_df[col]))
    train_df[col] = le.transform(train_df[col])
    test_df[col]  = le.transform(test_df[col])

y = train_df[TARGET].copy()
X = train_df.drop(columns=[TARGET]).copy()
test_features = test_df[X.columns].copy()   # keep identical column order

print('Encoding done. Feature matrix:', X.shape)

# =========================================================
# 5. CROSS-VALIDATED TRAINING (3 models, native categoricals,
#    in-fold target encoding, early stopping, log-target option)
# =========================================================
n_train, n_test = len(X), len(test_features)

oof_cat = np.zeros(n_train); test_cat = np.zeros(n_test)
oof_xgb = np.zeros(n_train); test_xgb = np.zeros(n_test)
oof_lgb = np.zeros(n_train); test_lgb = np.zeros(n_test)

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    print('\n' + '=' * 55)
    print(f'FOLD {fold + 1}')
    print('=' * 55)

    X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
    X_te       = test_features.copy()
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    # ---- leak-free OOF target encoding: fit on THIS fold's train only ----
    g_mean = y_tr.mean()
    for col, new_col in zip(te_cols, te_names):
        means = pd.DataFrame({'k': X_tr[col].values, 't': y_tr.values}) \
                  .groupby('k')['t'].mean()
        X_tr[new_col] = pd.Series(X_tr[col].values).map(means).fillna(g_mean).values
        X_va[new_col] = pd.Series(X_va[col].values).map(means).fillna(g_mean).values
        X_te[new_col] = pd.Series(X_te[col].values).map(means).fillna(g_mean).values

    # ---- target in model space ----
    y_tr_m = np.log1p(y_tr) if USE_LOG else y_tr.copy()
    y_va_m = np.log1p(y_va) if USE_LOG else y_va.copy()

    def back(p):
        p = np.expm1(p) if USE_LOG else p
        return np.clip(p, 0, None)

    cat_idx = [X_tr.columns.get_loc(c) for c in categorical_features]

    # ------------------- CatBoost (native cats) -------------------
    print('Training CatBoost...')
    m = CatBoostRegressor(
        iterations=5000, learning_rate=0.03, depth=8, l2_leaf_reg=3.0,
        loss_function='RMSE', eval_metric='RMSE',
        task_type='GPU', devices='0', random_seed=SEED, verbose=0)
    m.fit(X_tr, y_tr_m, eval_set=(X_va, y_va_m),
          cat_features=cat_idx, early_stopping_rounds=200, verbose=False)
    pv, pt = back(m.predict(X_va)), back(m.predict(X_te))
    oof_cat[va_idx] = pv; test_cat += pt / N_SPLITS
    print('  CatBoost  R2:', round(r2_score(y_va, pv), 5))

    # ------------------- XGBoost (numeric) -------------------
    print('Training XGBoost...')
    m = XGBRegressor(
        n_estimators=5000, learning_rate=0.02, max_depth=8,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_lambda=1.0, objective='reg:squarederror',
        tree_method='hist', device='cuda', random_state=SEED,
        early_stopping_rounds=200)
    m.fit(X_tr, y_tr_m, eval_set=[(X_va, y_va_m)], verbose=False)
    pv, pt = back(m.predict(X_va)), back(m.predict(X_te))
    oof_xgb[va_idx] = pv; test_xgb += pt / N_SPLITS
    print('  XGBoost   R2:', round(r2_score(y_va, pv), 5))

    # ------------------- LightGBM (native cats, fixed subsample) -------------------
    print('Training LightGBM...')
    m = LGBMRegressor(
        n_estimators=5000, learning_rate=0.02, num_leaves=63, max_depth=-1,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        reg_lambda=1.0, min_child_samples=20, random_state=SEED)
    m.fit(X_tr, y_tr_m, eval_set=[(X_va, y_va_m)],
          categorical_feature=categorical_features,
          callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)])
    pv, pt = back(m.predict(X_va)), back(m.predict(X_te))
    oof_lgb[va_idx] = pv; test_lgb += pt / N_SPLITS
    print('  LightGBM  R2:', round(r2_score(y_va, pv), 5))

# =========================================================
# 6. EVALUATE MODELS + OPTIMISE BLEND WEIGHTS ON OOF
# =========================================================
print('--- Single-model OOF R2 ---')
print('CatBoost :', round(r2_score(y, oof_cat), 5))
print('XGBoost  :', round(r2_score(y, oof_xgb), 5))
print('LightGBM :', round(r2_score(y, oof_lgb), 5))

oof_stack  = np.vstack([oof_cat,  oof_xgb,  oof_lgb]).T
test_stack = np.vstack([test_cat, test_xgb, test_lgb]).T

avg_r2 = r2_score(y, oof_stack.mean(axis=1))
print('\nSimple average OOF R2 :', round(avg_r2, 5))

def neg_r2(w):
    return -r2_score(y, oof_stack.dot(w))

res = minimize(neg_r2, x0=np.array([1/3] * 3), method='SLSQP',
               bounds=[(0.0, 1.0)] * 3,
               constraints=({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}))
w_opt  = res.x
opt_r2 = r2_score(y, oof_stack.dot(w_opt))
print('Optimised weights     :', np.round(w_opt, 4))
print('Optimised blend OOF R2:', round(opt_r2, 5))

# Use optimised weights only if they beat the simple average on OOF
if opt_r2 >= avg_r2:
    weights = w_opt
    print('\n>>> Using OPTIMISED weights')
else:
    weights = np.array([1/3] * 3)
    print('\n>>> Using SIMPLE AVERAGE (more robust here)')

# =========================================================
# 7. FINAL PREDICTIONS + SUBMISSION
# =========================================================
final_preds = np.clip(test_stack.dot(weights), 0, None)

original_test = pd.read_csv('/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv')
submission = pd.DataFrame({'Index': original_test['Index'], 'demand': final_preds})

out_path = '/content/drive/MyDrive/Traffic_Prediction/final_submission_Traffic_Prediction_optimized.csv'
submission.to_csv(out_path, index=False)

print(submission.head())
print('\nSaved ->', out_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete
Train: (77299, 37)  Test: (41778, 36)
Leaky / ID columns removed
Encoding done. Feature matrix: (77299, 31)

FOLD 1
Training CatBoost...
  CatBoost  R2: 0.95357
Training XGBoost...
  XGBoost   R2: 0.95648
Training LightGBM...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008292 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2076
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 35
[LightGBM] [Info] Start training from score 0.